In [ ]:
import torch, platform

print("Python           :", platform.python_version())
print("PyTorch          :", torch.__version__)
print("CUDA disponible  :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU              :", torch.cuda.get_device_name(0))
    print("Mémoire (Go)     :", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2))
else:
    print("⚠️  AUCUN GPU — activez : Exécution > Modifier le type d'exécution > GPU")

In [ ]:
!nvidia-smi

: 

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path

DRIVE_ROOT = Path('/content/drive/MyDrive/plant-disease')
for sub in ['dataset', 'models', 'results', 'results/figures']:
    (DRIVE_ROOT / sub).mkdir(parents=True, exist_ok=True)

print("Arborescence Drive prête :")
for p in sorted(DRIVE_ROOT.rglob('*')):
    print("  ", p.relative_to(DRIVE_ROOT))

In [ ]:
!pip install -q timm grad-cam

In [ ]:
import timm
print("timm :", timm.__version__)
print("Modèles ViT-Tiny disponibles :", timm.list_models('vit_tiny*', pretrained=True))

In [ ]:
# Option A — recommandée : cloner votre dépôt (après un premier push sur GitHub)
!git clone https://github.com/VOTRE_USER/plant-disease-classification.git /content/project
%cd /content/project

In [ ]:
import sys
sys.path.insert(0, '/content/project')

from src import config
from src.utils import set_seed, get_device

set_seed(config.SEED)
device = get_device()
print("Device      :", device)
print("Classes     :", config.NUM_CLASSES)
print("Dossier poids:", config.MODELS_DIR)

In [ ]:
import torch.nn as nn
import torch.optim as optim
from src.utils import save_checkpoint, load_checkpoint, file_size_mb
from src.metadata import build_metadata

# Modèle jouet, uniquement pour tester la mécanique de sauvegarde
dummy = nn.Linear(10, config.NUM_CLASSES)
opt   = optim.Adam(dummy.parameters(), lr=1e-3)

ckpt_path = config.MODELS_DIR / "_test_checkpoint.pt"
save_checkpoint(
    ckpt_path,
    model=dummy,
    optimizer=opt,
    epoch=0,
    history={"train_loss": [], "val_acc": []},
    metadata=build_metadata("dummy"),
)
print(f"Écrit  : {ckpt_path}  ({file_size_mb(ckpt_path):.3f} Mo)")

# Relecture
restored = nn.Linear(10, config.NUM_CLASSES)
ck = load_checkpoint(ckpt_path, model=restored)
print("Relu   :", ck["metadata"]["model_name"], "| classes :", ck["metadata"]["classes"])

ckpt_path.unlink()
print("Test OK, fichier temporaire supprimé.")